# Backtest Deep Dive

Use this notebook when a backtest result looks interesting (or suspicious) and
you want to understand *why* the metrics came out the way they did.

The HTML report gives you the summary. This notebook lets you drill into fills,
position behavior, regime dependence, and potential red flags.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from gnomepy_research.explore import load_results
from gnomepy_research.reporting.backtest import (
    compute_adverse_selection,
    plot_adverse_selection,
    compute_mm_stats,
    plot_mm_dashboard,
    compute_rolling_sharpe,
    compute_alpha_decay,
    detect_regimes,
    pnl_by_regime,
    plot_rolling_performance,
)
from gnomepy_research.analysis.signal_attribution import attribute_pnl_by_signal

## Configuration

In [ ]:
# Path to a backtest result directory
RESULTS_PATH = "gnomepy_research/sessions/n_exchange_arb/results/iter_015"

## 1. Load Results

In [ ]:
report = load_results(RESULTS_PATH)
summary = report.summary()

print(f"PnL:         ${summary.get('final_pnl', 0):,.2f}")
print(f"Sharpe:      {summary.get('sharpe', 0):.3f}")
print(f"Sortino:     {summary.get('sortino', 0):.3f}")
print(f"Fill count:  {summary.get('fill_count', 0):,}")
print(f"Total fees:  ${summary.get('total_fees', 0):,.4f}")
print(f"Market ticks:{summary.get('market_record_count', 0):,}")

## 2. PnL Analysis

Is the PnL smooth and consistent, or driven by a few lucky trades?

In [ ]:
report.plot_pnl().show()

In [ ]:
pnl = report.pnl_curve

# Drawdown curve
drawdown = pnl - pnl.cummax()
max_dd = drawdown.min()
max_dd_ts = drawdown.idxmin()

print(f"Max drawdown: ${max_dd:,.4f} at {max_dd_ts}")

# Distribution of 10-second bar returns
bar_returns = pnl.resample("10s").last().diff().dropna()
print(f"\n10s bar returns:")
print(f"  Mean:      ${bar_returns.mean():,.6f}")
print(f"  Std:       ${bar_returns.std():,.6f}")
print(f"  Skew:      {bar_returns.skew():.3f}")
print(f"  % positive: {(bar_returns > 0).mean():.1%}")

In [ ]:
# Zoom into the worst drawdown period
dd_end = max_dd_ts
# Walk backward to find where the drawdown started
before_dd = pnl[:dd_end]
dd_start = before_dd.idxmax()

zoom_pnl = pnl[dd_start:dd_end]
fig = go.Figure(go.Scatter(x=zoom_pnl.index, y=zoom_pnl.values, name="PnL"))
fig.update_layout(title=f"Worst Drawdown: {dd_start} → {dd_end}", height=350)
fig.show()

## 3. Fill Analysis

When does the strategy trade? What does the edge distribution look like?

In [ ]:
fills = report.fills
print(f"{len(fills)} fills")
fills.head()

In [ ]:
# Fill distribution by hour of day
if not fills.empty:
    fills_by_hour = fills.groupby(fills.index.hour).size()
    px.bar(fills_by_hour, title="Fill count by hour (UTC)", labels={"index": "Hour", "value": "Fills"}).show()

    # Edge per fill (mid - fill_price, sign-adjusted)
    if "fill_price" in fills.columns and "cash_flow" in fills.columns:
        px.histogram(
            fills["cash_flow"],
            nbins=50,
            title="Cash flow per fill distribution",
        ).show()

In [ ]:
# Adverse selection profile — does the market move against us after fills?
market_df = report._market_df
if not fills.empty and not market_df.empty:
    plot_adverse_selection(report).show()

## 4. Position Analysis

In [ ]:
report.plot_position().show()

In [ ]:
pos = report.position_curve
if not pos.empty:
    print(f"Max abs position: {pos.abs().max():.4f}")
    print(f"Mean abs position: {pos.abs().mean():.4f}")
    print(f"Time flat (pos≈0): {(pos.abs() < 1e-9).mean():.1%}")

## 5. Rolling Performance

Is the edge consistent across the trading session, or concentrated in specific windows?
Alpha decay means the signal is getting picked off over time.

In [ ]:
plot_rolling_performance(report).show()

## 6. Regime Analysis

Does the strategy only work in one market regime? If PnL is concentrated in
low-vol/tight-spread periods, it may not be robust.

In [ ]:
if not market_df.empty:
    regimes = detect_regimes(market_df)
    breakdown = pnl_by_regime(pnl, regimes)

    regime_df = pd.DataFrame(breakdown).T
    print("PnL by market regime:")
    print(regime_df.round(4))

## 7. Market Making Diagnostics

Skip this section for taker-only strategies.

In [ ]:
mm_stats = compute_mm_stats(report)
if mm_stats:
    print("Market making stats:")
    for k, v in mm_stats.items():
        print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")
    plot_mm_dashboard(report).show()
else:
    print("No passive quoting detected — this is a taker-only strategy.")

## 8. Signal Attribution

If the strategy recorded custom signal metrics, this section decomposes PnL
edge proportionally to each signal's magnitude at fill time.

Requires: strategy used `MetricRecorder` to log signal values tick-by-tick.

In [ ]:
custom = report.results.custom_metrics()
print("Custom metric buffers:", list(custom.keys()))

In [ ]:
# Example: if strategy recorded a 'signals' buffer with columns [mid_price, signal_a, signal_b]
# Uncomment and adapt to your strategy's metric buffer:

# SIGNAL_BUFFER = "signals"
# SIGNAL_COLUMNS = ["signal_a", "signal_b"]
# 
# if SIGNAL_BUFFER in custom and not fills.empty:
#     attr = attribute_pnl_by_signal(
#         fills_df=fills,
#         custom_metrics_df=custom[SIGNAL_BUFFER],
#         signal_columns=SIGNAL_COLUMNS,
#     )
#     print(attr.groupby(level=0)[[f"pnl_attr_{c}" for c in SIGNAL_COLUMNS]].mean().round(4))

## 9. Red Flags Checklist

Automated checks for common backtest artifacts.

In [ ]:
flags = []

# PnL concentration: top 10% of bars drive >80% of returns
bar_rets = pnl.resample("1min").last().diff().dropna()
if len(bar_rets) > 10:
    top10 = bar_rets.nlargest(max(1, len(bar_rets) // 10)).sum()
    total_positive = bar_rets[bar_rets > 0].sum()
    if total_positive > 0 and top10 / total_positive > 0.8:
        flags.append("⚠️  PnL heavily concentrated: top 10% of bars account for >80% of positive returns")

# Adverse selection > captured edge
if not fills.empty and not market_df.empty:
    ad = compute_adverse_selection(fills, market_df)
    if not ad.empty and "adverse_1s" in ad.columns:
        pct_adverse = (ad["adverse_1s"] < 0).mean()
        if pct_adverse > 0.55:
            flags.append(f"⚠️  High adverse selection: {pct_adverse:.0%} of fills move against us within 1s")

# Very few fills
if summary.get("fill_count", 0) < 20:
    flags.append(f"⚠️  Low fill count ({summary.get('fill_count')}): Sharpe estimate is unreliable")

# PnL only in one regime
if not market_df.empty:
    regime_pnls = {r: v.get("final_pnl", 0) for r, v in breakdown.items()}
    positive_regimes = sum(1 for v in regime_pnls.values() if v > 0)
    if positive_regimes <= 1 and len(regime_pnls) > 1:
        best_regime = max(regime_pnls, key=regime_pnls.get)
        flags.append(f"⚠️  PnL concentrated in one regime: {best_regime}")

if flags:
    print("Red flags found:")
    for f in flags:
        print(" ", f)
else:
    print("✓ No red flags detected")